# **NLP-RNN**
##### The goal of this project is to practice and learn how to use RNNs in natural language processing.
###### We write program in local jypyter notebook in VSCode. Then push it to a github repository. Finally we run our program on google colab.
#### **This program must be run on Google Colab.**

In [ ]:
# test system
import tensorflow as tf
print("Hello!")
print("Welcome to my program")
print(f"TensorFlow Version: {tf.__version__}")  

In [ ]:
# initialization

import os
import tensorflow as tf
import pathlib as path

# in local system: 
data_path=path.Path().resolve() / "../data/"

# in google colab: 
###     data_path = "/content/drive/MyDrive/NLP-RNN/"
# connect to google drive
###     from google.colab import drive
###     drive.mount('/content/drive')

# set vars
model_path = data_path / "my_shakespeare_model.keras"
epoch_file = data_path / "last_epoch.txt"
tep = 10 # total epochs

print("Model was saved in: ", model_path)
print("Epoch info was saved in: ", epoch_file)


### **Download and Preparing Data**

In [ ]:
# 1) Downloading Data: download all of Shakespeare’s work.

shakespeare_url = "https://homl.info/shakespeare" # shortcut URL
filepath = tf.keras.utils.get_file("shakespeare.txt", shakespeare_url)
with open(filepath) as f:
    shakespeare_text = f.read()

In [ ]:
# 2) Vectorization
text_vec_layer = tf.keras.layers.TextVectorization(split="character", standardize="lower")
text_vec_layer.adapt([shakespeare_text])
encoded = text_vec_layer([shakespeare_text])[0]

encoded -= 2 # drop tokens 0 (pad) and 1 (unknown), which we will not use
n_tokens = text_vec_layer.vocabulary_size() - 2 # number of distinct chars = 39
dataset_size = len(encoded) # total number of chars = 1,115,394

In [ ]:
# 3) Preparing Windows Dataset 
def to_dataset(sequence, length, shuffle=False, seed=None, batch_size=32):
    ds = tf.data.Dataset.from_tensor_slices(sequence)
    ds = ds.window(length + 1, shift=1, drop_remainder=True)
    ds = ds.flat_map(lambda window_ds: window_ds.batch(length + 1))
    if shuffle:
        ds = ds.shuffle(buffer_size=100_000, seed=seed)
    ds = ds.batch(batch_size)
    return ds.map(lambda window: (window[:, :-1], window[:, 1:])).prefetch(1)

In [ ]:
# 4) Create Datasets: training set, validation set, test set 
length = 100
tf.random.set_seed(42)
train_set = to_dataset(encoded[:1_000_000], length=length, shuffle=True,
seed=42)
valid_set = to_dataset(encoded[1_000_000:1_060_000], length=length)
test_set = to_dataset(encoded[1_060_000:], length=length)

### **Building, Training and Prediction**

In [ ]:
# 5) Model: building and training the NLP-RNN model

# 1- Checking the existence of the model file and the number of previous periods.
if os.path.exists(model_path) and os.path.exists(epoch_file):
    print("✅ Model and history file found. Load and continue learning...")
    
    # Loading the model
    model = tf.keras.models.load_model(model_path)
    model.compile(loss="sparse_categorical_crossentropy", optimizer="nadam", metrics=["accuracy"])
    
    # Reading the number of previous periods from the file
    with open(epoch_file, "r") as f:
        initial_epoch = int(f.read().strip())   
    total_epochs = tep
    
else:
    print("❌ No model or history found. Creating a new model from scratch...")
    
    # Building a new model
    model = tf.keras.Sequential([
                            tf.keras.layers.Embedding(input_dim=n_tokens, output_dim=16),
                            tf.keras.layers.GRU(128, return_sequences=True),
                            tf.keras.layers.Dense(n_tokens, activation="softmax")
                            ])
    model.compile(loss="sparse_categorical_crossentropy", optimizer="nadam", metrics=["accuracy"])
    
    initial_epoch = 0
    total_epochs = tep

# 2- Setting ModelCheckpoint
model_ckpt = tf.keras.callbacks.ModelCheckpoint(
                                        model_path,
                                        monitor="val_accuracy",
                                        save_best_only=True
                                        )

# 3- Custom callback to store the number of periods
class EpochSaver(tf.keras.callbacks.Callback):
    def on_epoch_end(self, epoch, logs=None):
        # Store the number of completed epochs (epoch + 1)
        with open(epoch_file, "w") as f:
            f.write(str(epoch + 1))

epoch_saver = EpochSaver()

# 4- Training implementation
print(f"🚀 Starting training from period {initial_epoch} to {total_epochs}...")
history = model.fit(
    train_set,
    validation_data=valid_set,
    epochs=total_epochs,
    initial_epoch=initial_epoch,
    callbacks=[model_ckpt, epoch_saver]  # Adding epoch_saver to callbacks
)

# 5- Making the final model
shakespeare_model = tf.keras.Sequential([
    text_vec_layer,
    tf.keras.layers.Lambda(lambda X: X - 2),
    model
])

In [ ]:
# 6) Prediction
y_proba = shakespeare_model.predict(tf.constant(["To be or not to b"]))[0, -1]
y_pred = tf.argmax(y_proba) # choose the most probable character ID
print(text_vec_layer.get_vocabulary()[y_pred + 2])

### **Generating Fake Shakespearean Text**

In [ ]:
# 7) example for text generation
log_probas = tf.math.log([[0.5, 0.4, 0.1]]) # probas = 50%, 40%, and 10%
tf.random.set_seed(42)
print(tf.random.categorical(log_probas, num_samples=8)) # draw 8 samples

In [ ]:
# 8) text genetator functions

def next_char(text, temperature=1):
    if isinstance(text, tf.Tensor):
        text = text.numpy().decode('utf-8')
    y_proba = shakespeare_model.predict(tf.constant([text]))  
    y_proba = y_proba[0, -1, :]  
    rescaled_logits = tf.math.log(y_proba + 1e-8) / temperature  
    char_id = tf.random.categorical(tf.expand_dims(rescaled_logits, 0), num_samples=1)[0, 0]
    return text_vec_layer.get_vocabulary()[char_id + 2]


def extend_text(text, n_chars=50, temperature=1):
    if isinstance(text, tf.Tensor):
        text = text.numpy().decode('utf-8')    
    for _ in range(n_chars):
        text += next_char(text, temperature)
    return text 

In [ ]:
# 9) text generate

class Color:
    RED = '\033[91m'
    GREEN = '\033[92m'
    YELLOW = '\033[93m'
    BLUE = '\033[94m'
    MAGENTA = '\033[95m'
    CYAN = '\033[96m'
    WHITE = '\033[97m'
    RESET = '\033[0m'   # بازگشت به حالت عادی
    BOLD = '\033[1m'    # پررنگ

tf.random.set_seed(42)
print(f"{Color.BLUE}_________________________________ temperature=0.01")
print(extend_text("To be or not to be", temperature=0.01))
print(extend_text("To be or not to be", temperature=1))
print(extend_text("To be or not to be", temperature=100))
